# FraudShield Advanced: Model Monitor via SDK

## Concept Review: Model Monitor Baselines

A baseline is a statistical profile of the training data that defines what
"normal" looks like for the model. Model Monitor uses the baseline as the
reference point for all subsequent monitoring. The baseline includes two
files stored in S3:

- **`statistics.json`** -- Per-feature statistics computed from training data:
  mean, standard deviation, min, max, median, distinct count, missing value
  percentages, and distribution histograms. These describe the expected
  distributions the model was trained on.

- **`constraints.json`** -- Rules derived from the statistics: data types,
  nullability, allowed value sets for categorical features, and acceptable
  ranges for numeric features. These are the thresholds that trigger violations
  when live data deviates.

Together, these files define the contract between the training data and
production inference. When incoming data breaks this contract -- a feature
that was never null starts arriving as null, or a numeric feature shifts
outside its trained range -- Model Monitor raises a violation.

This notebook is organized in three stages:

1. **Baselines and Data Capture** -- deploy an endpoint with data capture,
   generate a baseline from training data, inspect the outputs
2. **Scheduling and Violations** -- create monitoring schedules for data
   quality and model quality, examine violation reports
3. **Bias, Automation, and Cleanup** -- configure bias monitoring, connect
   monitors to automated retraining, clean up all resources

---

*This notebook runs on ml.t3.medium in SageMaker Studio. Monitoring jobs and
endpoints use ml.m5.xlarge (Free Tier eligible).*

In [ ]:
%pip install -q sagemaker boto3 pandas

import sagemaker
import boto3
import json
import time
import pandas as pd
from datetime import datetime, timedelta
from sagemaker.model_monitor import (
    DefaultModelMonitor,
    DataCaptureConfig,
    CronExpressionGenerator,
    ModelQualityMonitor,
)
from sagemaker.model_monitor.dataset_format import DatasetFormat
from sagemaker.xgboost import XGBoostModel

In [ ]:
sm_session = sagemaker.Session()
sm_client = sm_session.sagemaker_client
region = sm_session.boto_region_name
role = sagemaker.get_execution_role()
default_bucket = sm_session.default_bucket()

PREFIX = "fraudshield-monitoring"
ENDPOINT_NAME = f"fraudshield-monitor-ep-{datetime.now().strftime('%Y%m%d-%H%M')}"
CAPTURE_S3 = f"s3://{default_bucket}/{PREFIX}/data-capture"
BASELINE_S3 = f"s3://{default_bucket}/{PREFIX}/baseline"
REPORTS_S3 = f"s3://{default_bucket}/{PREFIX}/reports"
TRAIN_S3 = f"s3://{default_bucket}/{PREFIX}/processed/train.csv"
MODEL_S3 = f"s3://{default_bucket}/{PREFIX}/model/model.tar.gz"

print(f"Region         : {region}")
print(f"Role           : {role.split('/')[-1]}")
print(f"Default bucket : {default_bucket}")
print(f"Endpoint name  : {ENDPOINT_NAME}")

## Concept Review: Data Capture

Before monitoring can begin, the endpoint must capture inference requests.
Data capture samples incoming inference requests and/or responses and writes
them to S3 in JSON Lines format. Configuration options:

| Setting | Purpose |
|---|---|
| **Sampling percentage** | Fraction of requests to capture (1-100%). Use 100% for low-traffic endpoints, 10-20% for high-traffic to manage S3 costs |
| **Capture mode** | `Input` (requests only), `Output` (responses only), or `Input` and `Output` (both) |
| **S3 destination** | The S3 prefix where captured data files are stored |

Data capture must be enabled before any monitoring schedule can be created.
Without captured data, there is nothing for the monitoring job to compare
against the baseline.

In [ ]:
data_capture_config = DataCaptureConfig(
    enable_capture=True,
    sampling_percentage=100,
    destination_s3_uri=CAPTURE_S3,
    capture_options=["Input", "Output"],
    csv_content_types=["text/csv"],
)

xgb_model = XGBoostModel(
    model_data=MODEL_S3,
    role=role,
    framework_version="1.5-1",
    sagemaker_session=sm_session,
)

predictor = xgb_model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.xlarge",
    endpoint_name=ENDPOINT_NAME,
    data_capture_config=data_capture_config,
)

print(f"Endpoint deployed: {ENDPOINT_NAME}")

In [ ]:
endpoint_desc = sm_client.describe_endpoint(EndpointName=ENDPOINT_NAME)
endpoint_config_name = endpoint_desc["EndpointConfigName"]

config_desc = sm_client.describe_endpoint_config(
    EndpointConfigName=endpoint_config_name
)

dc_config = config_desc.get("DataCaptureConfig", {})
print(f"Endpoint Config  : {endpoint_config_name}")
print(f"Capture Enabled  : {dc_config.get('EnableCapture', False)}")
print(f"Sampling %       : {dc_config.get('InitialSamplingPercentage', 0)}")
print(f"Capture S3 URI   : {dc_config.get('DestinationS3Uri', 'N/A')}")
print(f"Capture Options  : {[o['CaptureMode'] for o in dc_config.get('CaptureOptions', [])]}")

In [ ]:
import numpy as np
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import CSVDeserializer

predictor.serializer = CSVSerializer()
predictor.deserializer = CSVDeserializer()

np.random.seed(42)
num_invocations = 50

for i in range(num_invocations):
    amount = np.random.uniform(10, 500)
    hour = np.random.randint(0, 24)
    category = np.random.randint(0, 5)
    age = np.random.randint(18, 75)
    distance = np.random.uniform(0, 100)

    payload = f"{amount:.2f},{hour},{category},{age},{distance:.2f}"
    response = predictor.predict(payload)

print(f"Sent {num_invocations} invocations to {ENDPOINT_NAME}")
print("Waiting 60 seconds for data capture files to appear in S3...")
time.sleep(60)

In [ ]:
s3_client = boto3.client("s3")

capture_prefix = f"{PREFIX}/data-capture/{ENDPOINT_NAME}"
response = s3_client.list_objects_v2(
    Bucket=default_bucket, Prefix=capture_prefix, MaxKeys=10
)

if "Contents" in response:
    print(f"Found {len(response['Contents'])} capture file(s):")
    for obj in response["Contents"]:
        print(f"  {obj['Key']}  ({obj['Size']} bytes)")
else:
    print("No capture files found yet. Wait and re-run this cell.")

## Concept Review: Creating a Baseline

The baselining job analyzes the training dataset and produces two files:

- **`statistics.json`** -- per-feature statistics: mean, standard deviation,
  min/max, unique value counts, missing value percentages, and distribution
  histograms.
- **`constraints.json`** -- rules derived from those statistics: data types,
  nullability, allowed value sets (categorical), and acceptable ranges
  (numeric).

The baseline job runs as a SageMaker Processing Job using the pre-built
Model Monitor container. Input is the training CSV in S3; output is the
statistics and constraints files written to a designated S3 prefix.

You can edit `constraints.json` after generation to tighten or relax rules
before attaching it to a schedule. For example, if the training data had a
maximum transaction amount of $2,000 but production could legitimately see
$5,000, you would increase the max constraint for that feature.

In [ ]:
my_monitor = DefaultModelMonitor(
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    volume_size_in_gb=20,
    max_runtime_in_seconds=3600,
)

my_monitor.suggest_baseline(
    baseline_dataset=TRAIN_S3,
    dataset_format=DatasetFormat.csv(header=True),
    output_s3_uri=BASELINE_S3,
    wait=True,
    logs=False,
)

print("Baseline job complete.")
print(f"Baseline artifacts: {BASELINE_S3}")

In [ ]:
baseline_stats_key = f"{PREFIX}/baseline/statistics.json"
baseline_constraints_key = f"{PREFIX}/baseline/constraints.json"

stats_obj = s3_client.get_object(Bucket=default_bucket, Key=baseline_stats_key)
stats = json.loads(stats_obj["Body"].read().decode("utf-8"))

print("=== Baseline Statistics (first 3 features) ===")
for feature in stats.get("features", [])[:3]:
    print(f"\nFeature: {feature.get('name', 'N/A')}")
    numerical = feature.get("numerical_statistics", {})
    if numerical:
        print(f"  Mean   : {numerical.get('mean', 'N/A')}")
        print(f"  StdDev : {numerical.get('std_dev', 'N/A')}")
        print(f"  Min    : {numerical.get('min', 'N/A')}")
        print(f"  Max    : {numerical.get('max', 'N/A')}")

print("\n=== Baseline Constraints (first 3 features) ===")
constraints_obj = s3_client.get_object(
    Bucket=default_bucket, Key=baseline_constraints_key
)
constraints = json.loads(constraints_obj["Body"].read().decode("utf-8"))

for feature in constraints.get("features", [])[:3]:
    print(f"\nFeature: {feature.get('name', 'N/A')}")
    print(f"  Type     : {feature.get('inferred_type', 'N/A')}")
    print(f"  Nullable : {feature.get('completeness', 'N/A')}")

---

## Stage 2: Scheduling and Violations

## Concept Review: Data Quality Monitoring

Data Quality Monitoring continuously compares live inference data against the
baseline to detect drift, missing values, schema violations, and out-of-range
values. The monitoring schedule runs a recurring Processing Job (hourly, daily,
or custom cron) that:

1. Reads captured data from S3
2. Computes the same statistics as the baseline job
3. Compares live statistics against the baseline constraints
4. Writes a `constraint_violations.json` report if any feature violates its
   constraints
5. Emits CloudWatch metrics (violation count, execution status)

Common violation types:

| Type | Meaning |
|---|---|
| `data_type_check` | Feature data type does not match baseline |
| `completeness_check` | Missing values exceed baseline threshold |
| `baseline_drift_check` | Feature distribution shifted significantly |
| `extra_column_check` | Column in live data not present in baseline |
| `missing_column_check` | Baseline column missing from live data |

In [ ]:
SCHEDULE_NAME = f"fraudshield-dq-schedule-{datetime.now().strftime('%Y%m%d-%H%M')}"

my_monitor.create_monitoring_schedule(
    monitor_schedule_name=SCHEDULE_NAME,
    endpoint_input=ENDPOINT_NAME,
    output_s3_uri=f"{REPORTS_S3}/data-quality",
    statistics=my_monitor.baseline_statistics(),
    constraints=my_monitor.suggested_constraints(),
    schedule_cron_expression=CronExpressionGenerator.hourly(),
    enable_cloudwatch_metrics=True,
)

print(f"Monitoring schedule created: {SCHEDULE_NAME}")

In [ ]:
schedule_desc = sm_client.describe_monitoring_schedule(
    MonitoringScheduleName=SCHEDULE_NAME
)

print(f"Schedule Name   : {schedule_desc['MonitoringScheduleName']}")
print(f"Schedule Status : {schedule_desc['MonitoringScheduleStatus']}")
print(f"Endpoint        : {schedule_desc['EndpointName']}")
print(f"Creation Time   : {schedule_desc['CreationTime']}")

schedule_config = schedule_desc["MonitoringScheduleConfig"]
print(f"Schedule Expr   : {schedule_config['ScheduleConfig']['ScheduleExpression']}")

### Data Quality vs. Model Quality: When Each Fires

| Scenario | Data Quality Violation? | Model Quality Violation? |
|---|---|---|
| Input distribution shifts, model still accurate | Yes | No |
| Input stable, but relationship between features and target changes (concept drift) | No | Yes |
| Both input drift and accuracy degradation | Yes | Yes |
| Drifted feature has low SHAP importance | Yes (but likely a false alarm) | No |

Data Quality violations are a leading indicator. Model Quality violations are
the definitive signal. Best practice: investigate Data Quality violations
immediately, but prioritize retraining decisions based on Model Quality results.

## Concept Review: Model Quality Monitoring

Data Quality Monitoring detects changes in the input data. Model Quality
Monitoring detects changes in prediction accuracy by comparing predictions
against ground truth labels.

The ground truth requirement is the key difference: you must have a mechanism
to collect actual outcomes and upload them to S3 matched to original prediction
request IDs. In fraud detection, ground truth is often delayed -- the true
outcome (fraudulent or not) may not be known for days or weeks.

Model Quality Monitoring computes metrics (accuracy, F1, AUC for classification;
RMSE, MAE for regression) on the labeled production data and compares them
against baseline metrics established during training evaluation. A violation
is raised when any metric degrades beyond a configurable threshold.

Model Quality violations are the strongest signal for retraining, because
they indicate the model's actual predictive performance has degraded, not
just that the input distribution has shifted.

In [ ]:
GROUND_TRUTH_S3 = f"s3://{default_bucket}/{PREFIX}/ground-truth"
MQ_BASELINE_S3 = f"s3://{default_bucket}/{PREFIX}/mq-baseline"
MQ_REPORTS_S3 = f"s3://{default_bucket}/{PREFIX}/reports/model-quality"
MQ_SCHEDULE_NAME = f"fraudshield-mq-schedule-{datetime.now().strftime('%Y%m%d-%H%M')}"

model_quality_monitor = ModelQualityMonitor(
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    volume_size_in_gb=20,
    max_runtime_in_seconds=1800,
    sagemaker_session=sm_session,
)

print(f"ModelQualityMonitor configured.")
print(f"Ground truth S3 path : {GROUND_TRUTH_S3}")
print(f"Schedule name        : {MQ_SCHEDULE_NAME}")
print()
print("To create the schedule, call:")
print("  model_quality_monitor.create_monitoring_schedule(")
print(f"      monitor_schedule_name='{MQ_SCHEDULE_NAME}',")
print(f"      endpoint_input='{ENDPOINT_NAME}',")
print(f"      ground_truth_input='{GROUND_TRUTH_S3}',")
print(f"      problem_type='BinaryClassification',")
print(f"      output_s3_uri='{MQ_REPORTS_S3}',")
print("      schedule_cron_expression=CronExpressionGenerator.hourly(),")
print("  )")
print()
print("Note: Model Quality requires ground truth labels uploaded to the")
print("ground_truth_input path before the monitoring job can compute metrics.")

## Concept Review: Drift Statistical Tests

When Model Monitor reports that a feature has "drifted," the underlying
mechanism is a statistical hypothesis test:

**Kolmogorov-Smirnov (K-S) test** -- used for continuous (numeric) features.
Compares the cumulative distribution functions (CDFs) of baseline and live
data. The K-S statistic (D) is the maximum absolute difference between the
two CDFs. D close to 0 means similar distributions; D close to 1 means
completely different distributions.

**Chi-Squared test** -- used for categorical features. Compares the frequency
distribution of categories in the baseline against live data. A large
Chi-Squared statistic means category proportions have changed.

Both tests produce a p-value. Model Monitor uses a default threshold of 0.05
(reject the null hypothesis of "no drift" when p < 0.05). You can adjust
thresholds per feature in `constraints.json`:

- Stricter threshold (0.01) for business-critical features like
  `transaction_amount`
- Relaxed threshold (0.10) for noisy features to avoid false alarms

An important nuance: with very large sample sizes, even trivially small
differences become statistically significant. Supplement p-values with the
effect size (the D statistic or Chi-Squared value itself) to determine
whether the drift is operationally meaningful.

In [ ]:
violations_prefix = f"{PREFIX}/reports/data-quality"

response = s3_client.list_objects_v2(
    Bucket=default_bucket, Prefix=violations_prefix
)

violation_files = []
if "Contents" in response:
    violation_files = [
        obj["Key"]
        for obj in response["Contents"]
        if "constraint_violations" in obj["Key"]
    ]

if violation_files:
    print(f"Found {len(violation_files)} violation report(s).")
    latest = sorted(violation_files)[-1]
    print(f"Latest: {latest}")
else:
    print("No violation reports found yet.")
    print("The monitoring schedule must execute at least once.")
    print("Check back after the next scheduled execution window.")

In [ ]:
if violation_files:
    latest_key = sorted(violation_files)[-1]
    v_obj = s3_client.get_object(Bucket=default_bucket, Key=latest_key)
    violations = json.loads(v_obj["Body"].read().decode("utf-8"))

    print(f"Total violations: {len(violations.get('violations', []))}")
    print()

    for v in violations.get("violations", []):
        print(f"Feature    : {v.get('feature_name', 'N/A')}")
        print(f"Type       : {v.get('constraint_check_type', 'N/A')}")
        print(f"Description: {v.get('description', 'N/A')}")
        print()
else:
    print("Skipping violation parsing -- no reports available yet.")
    print("\nExample violation structure:")
    example = {
        "violations": [
            {
                "feature_name": "transaction_amount",
                "constraint_check_type": "baseline_drift_check",
                "description": (
                    "Kolmogorov-Smirnov test: D=0.42, p-value=0.001. "
                    "Feature distribution has shifted significantly."
                ),
            }
        ]
    }
    print(json.dumps(example, indent=2))

---

## Stage 3: Bias, Automation, and Cleanup

## Concept Review: Bias and Attribution Drift

SageMaker Clarify integrates with Model Monitor to detect changes in model
fairness and feature importance over time:

**Bias Drift Monitoring** tracks fairness metrics across protected attributes
(e.g., gender, age group). A model that was fair at deployment can become
biased if data shifts disproportionately affect protected groups. Key metrics
include:

- **Demographic Parity (DPL):** Difference in positive prediction rates across
  groups
- **Treatment Equality (TE):** Whether error rates (FP/FN) are balanced across
  groups
- **Conditional Demographic Disparity (CDDL):** Demographic parity conditioned
  on legitimate factors

**Feature Attribution Drift** monitors whether the features driving predictions
have changed. Using SHAP values, it compares the global feature importance
ranking from training against live production data. If a feature like `zip_code`
starts dominating predictions when it was low-importance at training time, the
model may have developed a proxy for a protected attribute.

Both monitoring types require a Clarify baseline (bias metrics and SHAP values
computed on training data) and produce separate violation reports.

### Combining All Four Monitoring Types

In a production deployment, all four monitoring types run independently on the
same endpoint's captured data:

| Monitoring Type | Detects | Frequency | Baseline Source |
|---|---|---|---|
| Data Quality | Input distribution drift | Hourly or daily | Training data statistics |
| Model Quality | Prediction accuracy degradation | Weekly (depends on label availability) | Evaluation set metrics |
| Bias Drift | Fairness metric changes | Weekly or monthly | Clarify bias report |
| Feature Attribution | Feature importance changes | Weekly or monthly | Clarify SHAP report |

Each produces its own violation reports and CloudWatch metrics, enabling
targeted alerting per monitoring type. A data quality violation triggers
investigation; a model quality violation triggers retraining.

In [ ]:
all_schedules = sm_client.list_monitoring_schedules(
    EndpointName=ENDPOINT_NAME
)["MonitoringScheduleSummaries"]

print(f"Active monitoring schedules for {ENDPOINT_NAME}:")
print()
for s in all_schedules:
    print(f"  Name   : {s['MonitoringScheduleName']}")
    print(f"  Status : {s['MonitoringScheduleStatus']}")
    print(f"  Type   : {s.get('MonitoringType', 'DataQuality')}")
    print()

if not all_schedules:
    print("  No schedules found.")

In [ ]:
from sagemaker.model_monitor import ModelBiasMonitor, BiasAnalysisConfig
from sagemaker.clarify import (
    DataConfig,
    BiasConfig,
    ModelConfig,
    ModelPredictedLabelConfig,
)

BIAS_BASELINE_S3 = f"s3://{default_bucket}/{PREFIX}/bias-baseline"
BIAS_REPORTS_S3 = f"s3://{default_bucket}/{PREFIX}/reports/bias"

bias_config = BiasConfig(
    label_values_or_threshold=[1],
    facet_name="age_group",
    facet_values_or_threshold=[0],
)

model_bias_monitor = ModelBiasMonitor(
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    volume_size_in_gb=20,
    max_runtime_in_seconds=1800,
    sagemaker_session=sm_session,
)

print("ModelBiasMonitor configured.")
print(f"Protected attribute : age_group")
print(f"Baseline S3         : {BIAS_BASELINE_S3}")
print(f"Reports S3          : {BIAS_REPORTS_S3}")
print()
print("To create the bias monitoring schedule, first run a Clarify")
print("baseline job on training data, then call:")
print("  model_bias_monitor.create_monitoring_schedule(...)")
print("with the baseline output as the constraints reference.")

## Concept Review: Monitor and Pipeline Automation

Detection alone does not fix a degraded model. The closed-loop MLOps pattern
connects monitoring to automated remediation:

1. **Monitor** detects a violation (data drift, quality degradation, bias shift)
2. **CloudWatch Alarm** triggers based on violation metrics
3. **EventBridge Rule** matches the alarm state change event
4. **Lambda Function** calls `StartPipelineExecution` via the SageMaker SDK
5. **Pipeline** retrains the model on fresh data, evaluates, and registers a
   new version in Model Registry
6. **Approval gate** (manual or automatic) controls deployment
7. **Rebaseline** -- new baselines are computed from retraining data
8. Monitoring resumes with updated baselines

Safeguards against retraining loops:

- **Cooldown period** after each Pipeline execution (e.g., 24 hours)
- **Minimum improvement threshold** -- new model must exceed original baseline
  metrics, not just the degraded metrics
- **Maximum retrain frequency** -- limit Pipeline executions per time period

In [ ]:
alarm_config = {
    "AlarmName": "FraudShield-MonitorViolation-Alarm",
    "Namespace": "aws/sagemaker/Endpoints/data-metrics",
    "MetricName": "ViolationCount",
    "Dimensions": [
        {"Name": "MonitoringSchedule", "Value": SCHEDULE_NAME},
    ],
    "Statistic": "Sum",
    "Period": 3600,
    "EvaluationPeriods": 1,
    "Threshold": 0,
    "ComparisonOperator": "GreaterThanThreshold",
}

print("Example CloudWatch Alarm configuration:")
print(json.dumps(alarm_config, indent=2))
print()
print("This alarm triggers when the monitoring schedule detects any")
print("constraint violations. The alarm action would be an SNS topic")
print("or EventBridge rule that initiates the retraining pipeline.")

In [ ]:
cw_client = boto3.client("cloudwatch", region_name=region)

metrics_response = cw_client.list_metrics(
    Namespace="aws/sagemaker/Endpoints/data-metrics",
    MetricName="ViolationCount",
)

if metrics_response["Metrics"]:
    print("Model Monitor CloudWatch metrics found:")
    for m in metrics_response["Metrics"]:
        dims = {d["Name"]: d["Value"] for d in m["Dimensions"]}
        print(f"  Schedule: {dims.get('MonitoringSchedule', 'N/A')}")
else:
    print("No Model Monitor CloudWatch metrics found yet.")
    print("Metrics appear after the first monitoring execution completes.")

In [ ]:
eventbridge_rule_pattern = {
    "source": ["aws.cloudwatch"],
    "detail-type": ["CloudWatch Alarm State Change"],
    "detail": {
        "alarmName": ["FraudShield-MonitorViolation-Alarm"],
        "state": {
            "value": ["ALARM"]
        },
    },
}

print("Example EventBridge rule pattern for triggering retraining:")
print(json.dumps(eventbridge_rule_pattern, indent=2))
print()
print("This pattern matches when the CloudWatch alarm for monitor violations")
print("transitions to the ALARM state. The EventBridge target would be a")
print("Lambda function that calls sagemaker:StartPipelineExecution.")

---

## Mandatory Cleanup

Delete all resources created in this notebook to avoid ongoing charges.
Cleanup order matters: delete monitoring schedules before deleting the
endpoint, because a schedule cannot execute against a deleted endpoint
and will produce errors in CloudWatch.

In [ ]:
print("=== Cleanup: Deleting monitoring schedules ===")
try:
    my_monitor.delete_monitoring_schedule()
    print(f"Deleted data quality schedule: {SCHEDULE_NAME}")
except Exception as e:
    print(f"Data quality schedule deletion: {e}")

print()
print("=== Cleanup: Deleting endpoint ===")
try:
    predictor.delete_endpoint(delete_endpoint_config=True)
    print(f"Deleted endpoint: {ENDPOINT_NAME}")
except Exception as e:
    print(f"Endpoint deletion: {e}")

print()
print("=== Cleanup: Deleting model ===")
try:
    predictor.delete_model()
    print("Deleted model.")
except Exception as e:
    print(f"Model deletion: {e}")

print()
print("=== Verifying cleanup ===")
try:
    sm_client.describe_endpoint(EndpointName=ENDPOINT_NAME)
    print(f"WARNING: Endpoint {ENDPOINT_NAME} still exists.")
except sm_client.exceptions.ClientError:
    print(f"Endpoint {ENDPOINT_NAME} confirmed deleted.")

schedules = sm_client.list_monitoring_schedules(
    EndpointName=ENDPOINT_NAME
)["MonitoringScheduleSummaries"]
if schedules:
    print(f"WARNING: {len(schedules)} schedule(s) still active.")
else:
    print("All monitoring schedules confirmed deleted.")

print()
print("Cleanup complete. Verify no resources remain in the SageMaker console.")